In [ ]:
import json
import requests
from pathlib import Path
with open(r"C:\Users\Abhinav Arora\Desktop\COCO custom model\validation_data_COCO19.json", "r") as file:
    data = json.load(file)

image_ids = []
for a in data["annotations"]:
    image_ids.append(a["image_id"])

def download_coco_images(image_ids, output_dir, split="train2017"):
    """
    image_ids: list of integer image ids
    output_dir: folder to save images into
    split: "train2017" or "val2017"
    """
    base_url = f"http://images.cocodataset.org/{split}"
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    total = len(image_ids)
    failed = []

    for i, image_id in enumerate(image_ids):
        # Zero-pad to 12 digits
        filename = f"{str(image_id).zfill(12)}.jpg"
        save_path = output_path / filename

        # Skip if already downloaded
        if save_path.exists():
            print(f"[{i+1}/{total}] Skipping {filename} (already exists)")
            continue

        url = f"{base_url}/{filename}"

        try:
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                with open(save_path, "wb") as f:
                    f.write(response.content)
                print(f"[{i+1}/{total}] Downloaded {filename}")
            else:
                print(f"[{i+1}/{total}] Failed {filename} — status {response.status_code}")
                failed.append(image_id)
        except Exception as e:
            print(f"[{i+1}/{total}] Error {filename} — {e}")
            failed.append(image_id)

    print(f"\nDone. {total - len(failed)}/{total} downloaded successfully.")
    if failed:
        print(f"Failed image ids: {failed}")

# Usage
download_coco_images(image_ids, output_dir="coco_images_validation", split="val2017")